# Import packages

Quick memory chceks:

In [1]:
# to prevent blocking all memory
import os
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

import tensorflow as tf
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

print("TF GPUs:", tf.config.list_physical_devices("GPU"))

2026-03-12 15:38:21.150967: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-12 15:38:25.332229: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-12 15:38:42.845561: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TF GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
import cv2
from keras.utils import load_img
from keras.saving import load_model
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from skimage.measure import regionprops, regionprops_table
from tqdm import trange, tqdm

import segmenteverygrain as seg
import segmenteverygrain.interactions as si
import os
import rasterio
import time, traceback
import pandas as pd
import torch

# %matplotlib qt

## Load models

Maybe you need to clone first SAM2. Then add the yaml file in the bottom of the next chunk

For terrabyte:

In [3]:
import torch

# 1) UNet laden (TF)
unet = load_model(
    "../models/seg_model.keras",
    custom_objects={"weighted_crossentropy": seg.weighted_crossentropy},
)
print("UNet loaded")

# 2) SAM2 laden (Torch)
device = "cuda" if torch.cuda.is_available() else "cpu"
cfg  = "configs/sam2.1/sam2.1_hiera_l.yaml"
ckpt = "/dss/dsshome1/0B/di54doz/segmenteverygrain/models/sam2.1_hiera_large.pt"
sam = build_sam2(cfg, ckpt, device=device)
print("SAM2 loaded on", device)

# 3) kurzer Speicher-Check
free, total = torch.cuda.mem_get_info()
print(f"torch reports free {free/1024**3:.1f} GB / total {total/1024**3:.1f} GB")

I0000 00:00:1773326375.478724   91170 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79193 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:4b:00.0, compute capability: 8.0


UNet loaded
SAM2 loaded on cuda
torch reports free 77.7 GB / total 79.3 GB


For private Laptop:

In [ ]:
# UNET model
unet = load_model(
    "../models/seg_model.keras",
    custom_objects={"weighted_crossentropy": seg.weighted_crossentropy},
)
print("UNet loaded")

# Auto-detect device: CUDA for NVIDIA, MPS for Apple Silicon, CPU as fallback
import torch
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"



In [ ]:
# # SAM 2.1 checkpoints. Download from:
# # https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

# sam = build_sam2(r"C:\Users\leoni\sam2\sam2\configs\sam2.1\sam2.1_hiera_l.yaml", r"C:\Users\leoni\segmenteverygrain\models\sam2.1_hiera_large.pt", device=device)


## Funktion for processing one folder: 

In [4]:
import os
import glob
import time
import traceback
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.utils import load_img
import rasterio

import pandas as pd
import torch

import geopandas as gpd
from shapely.geometry import Polygon
from skimage.measure import regionprops_table

# -----------------------------
# USER SETTINGS
# -----------------------------
OUT_ROOT = "/dss/tbyscratch/0B/di54doz/seg"   # alles hier rein
os.makedirs(OUT_ROOT, exist_ok=True)

MAX_SAM_PROMPTS = 3000
PROMPT_SUBSAMPLE_MODE = "random"   # "random" oder "first"
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

# Save label mask TIFF always
SAVE_LABEL_TIF = True

# Keep plots
SAVE_PEBBLE_PNG = True

# Histogram only for sample (saves a lot of time)
SAVE_HIST_SAMPLE = True
N_HIST_PER_FOLDER = 100  # random tiles per folder for histogram PNGs

# Save GPKG only for sample
SAVE_GPKG_SAMPLE = True
N_GPKG_PER_FOLDER = 100  # random tiles per folder

# SEG plotting: avoid drawing ellipse axes from segmenteverygrain
SEG_PLOT_IMAGE = False   # <- important: prevents a/b axes overlays

def _gpu_free_gb():
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        return free / 1024**3
    return None

def setup_out_folder(in_folder: str, out_root: str):
    folder_name = os.path.basename(os.path.normpath(in_folder))
    out_folder = os.path.join(out_root, folder_name)

    out_gpkg   = os.path.join(out_folder, "ouputgpkg")
    out_csv    = os.path.join(out_folder, "csv_stats")
    out_plot   = os.path.join(out_folder, "pebbleplots")
    out_hist   = os.path.join(out_folder, "histplots")
    out_masks  = os.path.join(out_folder, "masktifs")

    for p in [out_gpkg, out_csv, out_plot, out_hist, out_masks]:
        os.makedirs(p, exist_ok=True)

    return out_folder, out_gpkg, out_csv, out_plot, out_hist, out_masks

def collect_tiles(folder: str):
    inputtiledir = os.path.join(folder, "inputtiles")
    tiles_in_input = sorted(glob.glob(os.path.join(inputtiledir, "*.tif")))
    tiles_in_root  = sorted(glob.glob(os.path.join(folder, "*.tif")))
    tiles = tiles_in_input if len(tiles_in_input) > 0 else tiles_in_root

    source = "inputtiles" if len(tiles_in_input) > 0 else "folder root"
    print(f"Found {len(tiles)} tiles in {folder} ({source})")

    if len(tiles) == 0:
        raise RuntimeError(f"No tiles found in {inputtiledir} or in {folder}")

    return tiles

def process_one_folder(folder: str, out_root: str = OUT_ROOT):
    tiles = collect_tiles(folder)

    out_folder, out_gpkg, out_csv, out_plot, out_hist, out_masks = setup_out_folder(folder, out_root)
    print("OUT_FOLDER:", out_folder)

    tile_ids_all = [os.path.splitext(os.path.basename(p))[0] for p in tiles]

    # sample for GPKG export
    if SAVE_GPKG_SAMPLE:
        if len(tile_ids_all) <= N_GPKG_PER_FOLDER:
            gpkg_tile_ids = set(tile_ids_all)
        else:
            gpkg_tile_ids = set(rng.choice(tile_ids_all, size=N_GPKG_PER_FOLDER, replace=False))
        print(f"GPKG sample: {len(gpkg_tile_ids)} / {len(tile_ids_all)} tiles")
    else:
        gpkg_tile_ids = set()

    # sample for histogram plots
    if SAVE_HIST_SAMPLE:
        if len(tile_ids_all) <= N_HIST_PER_FOLDER:
            hist_tile_ids = set(tile_ids_all)
        else:
            hist_tile_ids = set(rng.choice(tile_ids_all, size=N_HIST_PER_FOLDER, replace=False))
        print(f"HIST sample: {len(hist_tile_ids)} / {len(tile_ids_all)} tiles")
    else:
        hist_tile_ids = set()

    # read pixel size once from first tile
    with rasterio.open(tiles[0]) as src:
        xres, yres = src.res
        crs = src.crs

    gsd_units = float((xres + yres) / 2.0)
    gsd_mm = gsd_units * 1000.0

    minor_mm = 20.0
    minor_px = minor_mm / gsd_mm
    min_area_px = float(np.pi * (minor_px / 2.0) ** 2)

    print("CRS:", crs)
    print(f"Pixel size: {gsd_units:.6f} units/px (~{gsd_mm:.3f} mm/px)")
    print(f"2 cm => minor_px={minor_px:.2f} px -> min_area_px={min_area_px:.1f} px^2")

    rows = []
    metrics_csv = os.path.join(out_folder, "runtime_metrics.csv")
    pd.DataFrame([]).to_csv(metrics_csv, index=False)  # creates empty file early

    for i, fname in enumerate(tiles, start=1):
        tile_id = os.path.splitext(os.path.basename(fname))[0]
        print(f"[{i}/{len(tiles)}] {tile_id}")
        with open(os.path.join(out_folder, "_progress.txt"), "w") as f:
            f.write(f"{i}/{len(tiles)} {tile_id}\n")

        t0 = time.perf_counter()
        gpu_free_before = _gpu_free_gb()

        rec = {
            "folder": os.path.basename(folder),
            "tile_id": tile_id,
            "fname": fname,
            "status": None,

            "crs": str(crs),
            "pixel_size_units_per_px": gsd_units,
            "pixel_size_mm_per_px": gsd_mm,
            "minor_mm_threshold": minor_mm,
            "minor_px_threshold": minor_px,
            "min_area_px": min_area_px,

            "H": None,
            "W": None,
            "n_prompts_before": None,
            "n_prompts_used": None,
            "prompt_cap_used": None,
            "n_grains": None,

            "t_unet_s": None,
            "t_label_s": None,
            "t_sam_s": None,
            "t_export_s": None,
            "t_total_s": None,

            "gpu_free_gb_before": gpu_free_before,
            "gpu_free_gb_after": None,

            "error_msg": None,
            "traceback_head": None,
        }

        try:
            # nodata check
            with rasterio.open(fname) as src:
                m = src.dataset_mask()
                if not np.any(m):
                    print(" -> skipped (100% Nodata)")
                    rec["status"] = "skip_nodata"
                    rec["t_total_s"] = time.perf_counter() - t0
                    rec["gpu_free_gb_after"] = _gpu_free_gb()
                    rows.append(rec)
                    continue

            # load + predict (UNet)
            t = time.perf_counter()
            image = np.array(load_img(fname))
            rec["H"], rec["W"] = int(image.shape[0]), int(image.shape[1])
            image_pred = seg.predict_image(image, unet, I=256)
            rec["t_unet_s"] = time.perf_counter() - t

            # prompts
            t = time.perf_counter()
            labels_pts, coords = seg.label_grains(image, image_pred, dbs_max_dist=10.0)
            rec["t_label_s"] = time.perf_counter() - t

            coords = np.asarray(coords)
            rec["n_prompts_before"] = int(len(coords))
            rec["prompt_cap_used"] = False

            if MAX_SAM_PROMPTS is not None and len(coords) > MAX_SAM_PROMPTS:
                rec["prompt_cap_used"] = True
                n_before = len(coords)

                if PROMPT_SUBSAMPLE_MODE == "first":
                    keep_idx = np.arange(MAX_SAM_PROMPTS)
                else:
                    keep_idx = np.sort(rng.choice(len(coords), size=MAX_SAM_PROMPTS, replace=False))

                coords = coords[keep_idx]

                try:
                    labels_arr = np.asarray(labels_pts)
                    if labels_arr.ndim == 1 and len(labels_arr) == n_before:
                        labels_pts = labels_arr[keep_idx]
                except Exception:
                    pass

                print(f"Prompt cap active: reduced prompts from {n_before} -> {len(coords)}")

            rec["n_prompts_used"] = int(len(coords))

            # SAM segmentation
            t = time.perf_counter()
            all_grains, labels, mask_all, grain_data, fig, ax = seg.sam_segmentation(
                sam,
                image,
                image_pred,
                coords,
                labels_pts,
                min_area=min_area_px,
                plot_image=SEG_PLOT_IMAGE,   # <- no built-in axis overlays
                remove_edge_grains=True,
                remove_large_objects=True,
            )
            rec["t_sam_s"] = time.perf_counter() - t
            rec["n_grains"] = int(len(all_grains))

            # export/post
            t = time.perf_counter()

            # 1) pebble PNG (fallback plot, no axes, no a/b axis overlays)
            if SAVE_PEBBLE_PNG:
                seg_plot_path = os.path.join(out_plot, f"{tile_id}.png")

                if fig is None:
                    fig, ax = plt.subplots(figsize=(8, 8))
                    ax.imshow(image)
                    for poly in all_grains:
                        x, y = poly.exterior.xy
                        ax.plot(x, y, linewidth=0.8)
                    ax.axis("off")
                else:
                    try:
                        for a in fig.axes:
                            a.axis("off")
                    except Exception:
                        pass

                fig.savefig(seg_plot_path, dpi=200, bbox_inches="tight")
                plt.close(fig)

            # 2) label mask GeoTIFF (0..N), georeferenced, filtered result from SEG
            if SAVE_LABEL_TIF:
                with rasterio.open(fname) as ds:
                    prof = ds.profile.copy()
                    prof.update(driver="GTiff", count=1, dtype="uint32", compress="lzw")
                    out_mask = os.path.join(out_masks, f"{tile_id}_labels.tif")
                    with rasterio.open(out_mask, "w", **prof) as dst:
                        dst.write(labels.astype("uint32"), 1)

            # 3) stats from labels (fast)
            with rasterio.open(fname) as dataset:
                px_size_x = abs(dataset.transform.a)

                props = regionprops_table(
                    labels.astype("int"),
                    properties=("label", "area", "centroid", "major_axis_length", "minor_axis_length"),
                )
                grain_stats = pd.DataFrame(props)

                if len(grain_stats) > 0:
                    centroid_x, centroid_y = rasterio.transform.xy(
                        dataset.transform,
                        grain_stats["centroid-0"].values,
                        grain_stats["centroid-1"].values,
                    )

                    grain_stats["centroid_x"] = centroid_x
                    grain_stats["centroid_y"] = centroid_y
                    grain_stats.rename(columns={"area": "area_px"}, inplace=True)
                    grain_stats["major_axis_px"] = grain_stats["major_axis_length"]
                    grain_stats["minor_axis_px"] = grain_stats["minor_axis_length"]
                    grain_stats["major_axis_length"] = grain_stats["major_axis_length"] * px_size_x
                    grain_stats["minor_axis_length"] = grain_stats["minor_axis_length"] * px_size_x
                    grain_stats["major_axis_mm"] = grain_stats["major_axis_length"] * 1000.0
                    grain_stats["minor_axis_mm"] = grain_stats["minor_axis_length"] * 1000.0

                    out_cols = [
                        "label", "area_px", "centroid_x", "centroid_y",
                        "major_axis_px", "minor_axis_px",
                        "major_axis_length", "minor_axis_length",
                        "major_axis_mm", "minor_axis_mm",
                    ]
                    grain_stats_out = grain_stats[out_cols].copy()
                else:
                    grain_stats_out = pd.DataFrame(columns=[
                        "label","area_px","centroid_x","centroid_y",
                        "major_axis_px","minor_axis_px",
                        "major_axis_length","minor_axis_length",
                        "major_axis_mm","minor_axis_mm"
                    ])

            csv_path = os.path.join(out_csv, f"{tile_id}_grain_stats.csv")
            grain_stats_out.to_csv(csv_path, index=False)

            # 4) histogram PNG only for sampled tiles
            if SAVE_HIST_SAMPLE and (tile_id in hist_tile_ids) and len(grain_stats_out) > 0:
                fig_hist, ax_hist = seg.plot_histogram_of_axis_lengths(
                    grain_stats_out["major_axis_mm"],
                    grain_stats_out["minor_axis_mm"],
                    binsize=0.25,
                    xlimits=[8, 2 * 256],
                )
                hist_plot_path = os.path.join(out_hist, f"{tile_id}_hist.png")
                fig_hist.savefig(hist_plot_path, dpi=200, bbox_inches="tight")
                plt.close(fig_hist)

            # 5) GPKG only for sampled tiles
            if SAVE_GPKG_SAMPLE and (tile_id in gpkg_tile_ids):
                with rasterio.open(fname) as dataset:
                    projected_polys = []
                    for grain in all_grains:
                        px_x = np.asarray(grain.exterior.xy[0])
                        px_y = np.asarray(grain.exterior.xy[1])
                        x_proj, y_proj = rasterio.transform.xy(dataset.transform, px_y, px_x)
                        poly = Polygon(np.vstack((x_proj, y_proj)).T)
                        projected_polys.append(poly)

                    gdf = gpd.GeoDataFrame({"geometry": projected_polys}, geometry="geometry", crs=dataset.crs)

                gpkg_path = os.path.join(out_gpkg, f"{tile_id}_grains.gpkg")
                gdf.to_file(gpkg_path, driver="GPKG")

            rec["t_export_s"] = time.perf_counter() - t
            rec["status"] = "ok"
            rec["t_total_s"] = time.perf_counter() - t0
            rec["gpu_free_gb_after"] = _gpu_free_gb()
            rows.append(rec)
            pd.DataFrame(rows).to_csv(metrics_csv, index=False)

        except Exception as e:
            rec["status"] = "error"
            rec["t_total_s"] = time.perf_counter() - t0
            rec["gpu_free_gb_after"] = _gpu_free_gb()
            rec["error_msg"] = str(e)
            rec["traceback_head"] = traceback.format_exc(limit=8)
            rows.append(rec)
            pd.DataFrame(rows).to_csv(metrics_csv, index=False)
            print("ERROR on", tile_id, ":", e)

    # runtime metrics for this folder
    df = pd.DataFrame(rows)
    metrics_csv = os.path.join(out_folder, "runtime_metrics.csv")
    df.to_csv(metrics_csv, index=False)
    print("Saved runtime metrics CSV:", metrics_csv)

    # ready table
    ok = df[df["status"] == "ok"].copy()
    n_ok = len(ok)
    total_s = ok["t_total_s"].sum()
    tiles_per_min = (n_ok / (total_s / 60.0)) if total_s > 0 else np.nan

    ready = pd.DataFrame({
        "metric": [
            "n_tiles_total",
            "n_tiles_ok",
            "n_tiles_skipped_nodata",
            "n_tiles_error",
            "total_runtime_min",
            "tiles_per_min",
            "total_s_per_tile (median)",
            "unet_s (median)",
            "label_s (median)",
            "sam_s (median)",
            "export_s (median)",
            "prompts_used (median)",
            "grains (median)",
        ],
        "value": [
            int(len(df)),
            int(n_ok),
            int((df["status"] == "skip_nodata").sum()),
            int((df["status"] == "error").sum()),
            float(total_s / 60.0) if n_ok else np.nan,
            float(tiles_per_min) if n_ok else np.nan,
            float(ok["t_total_s"].median()) if n_ok else np.nan,
            float(ok["t_unet_s"].median()) if n_ok else np.nan,
            float(ok["t_label_s"].median()) if n_ok else np.nan,
            float(ok["t_sam_s"].median()) if n_ok else np.nan,
            float(ok["t_export_s"].median()) if n_ok else np.nan,
            float(ok["n_prompts_used"].median()) if n_ok else np.nan,
            float(ok["n_grains"].median()) if n_ok else np.nan,
        ]
    })

    ready_csv = os.path.join(out_folder, "runtime_summary_ready_table.csv")
    ready.to_csv(ready_csv, index=False)
    print("Saved ready table CSV:", ready_csv)

    return df, ready

# Set folder strucure

In [ ]:
# import os, glob

# BASE = "/dss/dsstbyfs02/pr94no/pr94no-dss-0001/drylands/gravel_leonie_masterthesis/segmenteverygrain"
# subdirs = ["inputtiles", "ouputgpkg", "pebbleplots", "histplots", "csv_stats"]

# folders = sorted(glob.glob(os.path.join(BASE, "F*")))
# print("Found F folders:", [os.path.basename(f) for f in folders])

# for folder in folders:
#     made_any = False
#     for sd in subdirs:
#         path = os.path.join(folder, sd)
#         if not os.path.isdir(path):
#             os.makedirs(path, exist_ok=True)
#             made_any = True
#     if made_any:
#         print("Initialized structure:", os.path.basename(folder))
#     else:
#         print("OK already structured:", os.path.basename(folder))

# Run segmentation

## Not including Masks and saving all statistiks, plots and images and gpkgs

For only one folder:

In [ ]:
# process_one_folder("/dss/dsstbyfs02/pr94no/pr94no-dss-0001/drylands/gravel_leonie_masterthesis/segmenteverygrain/F9")

For processing over several folders (F-folders):

In [5]:
import os, glob

BASE = "/dss/dsstbyfs02/pr94no/pr94no-dss-0001/drylands/gravel_leonie_masterthesis/segmenteverygrain"
folders = sorted(glob.glob(os.path.join(BASE, "F*")))

MAX_TO_PROCESS = 20
processed = 0

for folder in folders:
    if processed >= MAX_TO_PROCESS:
        print("Reached MAX_TO_PROCESS, stopping.")
        break

    # 1) skip wenn schon Ergebnisse (GPKG) vorhanden
    gpkg_files = glob.glob(os.path.join(folder, "ouputgpkg", "*.gpkg"))
    if len(gpkg_files) > 0:
        print("SKIP already has GPKG:", os.path.basename(folder), "n_gpkg:", len(gpkg_files))
        continue

    # 2) skip wenn noch keine tiles drin sind (inputtiles/ ODER folder-root)
    tiles = glob.glob(os.path.join(folder, "inputtiles", "*.tif"))
    if len(tiles) == 0:
        tiles = glob.glob(os.path.join(folder, "*.tif"))

    if len(tiles) == 0:
        print("SKIP no tiles yet:", os.path.basename(folder))
        continue

    print("PROCESS:", os.path.basename(folder), "tiles:", len(tiles))
    process_one_folder(folder)
    processed += 1

PROCESS: F23 tiles: 997
Found 997 tiles in /dss/dsstbyfs02/pr94no/pr94no-dss-0001/drylands/gravel_leonie_masterthesis/segmenteverygrain/F23 (folder root)
OUT_FOLDER: /dss/tbyscratch/0B/di54doz/seg/F23
GPKG sample: 100 / 997 tiles
HIST sample: 100 / 997 tiles
CRS: EPSG:32643
Pixel size: 0.002877 units/px (~2.877 mm/px)
2 cm => minor_px=6.95 px -> min_area_px=38.0 px^2
[1/997] tile_r000004_c000007
segmenting image tiles...


100%|██████████| 4/4 [00:01<00:00,  2.53it/s]


creating masks using SAM...


100%|██████████| 81/81 [00:05<00:00, 15.44it/s]


finding overlapping polygons...


...

creating labeled image...
[699/997] tile_r000019_c000042
segmenting image tiles...


100%|██████████| 4/4 [00:01<00:00,  2.51it/s]


creating masks using SAM...


  8%|▊         | 58/746 [00:02<00:25, 26.79it/s]


KeyboardInterrupt: 

# Plotting raster like in IG2

In [ ]:
# Pretty overlays for SEG label masks
# Makes a random sample of PNGs per F-folder, using your palette + black outline.
# Reads:
#   SEG outputs: OUT_ROOT/<Fxx>/masktifs/*_labels.tif
#   Original tiles: SEG_BASE/<Fxx>/inputtiles/*.tif  (fallback: SEG_BASE/<Fxx>/*.tif)
# Writes:
#   OUT_ROOT/<Fxx>/pebbleplots_pretty/<Fxx>_<tile_id>_pretty.png

import os, glob
import numpy as np
import rasterio
import matplotlib.pyplot as plt

from skimage.segmentation import find_boundaries
from matplotlib.colors import ListedColormap

# -----------------------------
# SETTINGS
# -----------------------------
SEG_BASE = "/dss/dsstbyfs02/pr94no/pr94no-dss-0001/drylands/gravel_leonie_masterthesis/segmenteverygrain"
OUT_ROOT = "/dss/tbyscratch/0B/di54doz/seg"   # your SEG OUT_ROOT from segmentation runs

MAX_FOLDERS = None          # None = all
SAMPLE_PER_FOLDER = 25      # how many overlays per folder
RNG_SEED = 42

OVERLAY_ALPHA = 0.45
OUTLINE_ALPHA = 0.95
DPI = 200

PALETTE_HEX = ["#065143", "#77CBB9", "#5B2E48", "#F96900", "#F4D35E"]

# -----------------------------
# HELPERS
# -----------------------------
def hex_to_rgba(h, a=1.0):
    h = h.lstrip("#")
    r = int(h[0:2], 16) / 255.0
    g = int(h[2:4], 16) / 255.0
    b = int(h[4:6], 16) / 255.0
    return (r, g, b, a)

PALETTE_RGBA = [hex_to_rgba(h, 1.0) for h in PALETTE_HEX]

def make_palette_cmap(n_labels, rng, alpha=1.0):
    colors = [(0, 0, 0, 0)]  # transparent background
    for _ in range(n_labels):
        c = list(PALETTE_RGBA[int(rng.integers(0, len(PALETTE_RGBA)))])
        c[3] = alpha
        colors.append(tuple(c))
    return ListedColormap(colors)

def find_tile_dir(seg_folder: str) -> str:
    cand = os.path.join(seg_folder, "inputtiles")
    if os.path.isdir(cand) and len(glob.glob(os.path.join(cand, "*.tif"))) > 0:
        return cand
    return seg_folder

# -----------------------------
# RUN
# -----------------------------
folders = sorted(glob.glob(os.path.join(SEG_BASE, "F*")))
rng = np.random.default_rng(RNG_SEED)

done_folders = 0
done_pngs_total = 0

for folder in folders:
    if MAX_FOLDERS is not None and done_folders >= MAX_FOLDERS:
        print("Reached MAX_FOLDERS, stopping.")
        break

    folder_name = os.path.basename(os.path.normpath(folder))
    out_dir = os.path.join(OUT_ROOT, folder_name)
    mask_dir = os.path.join(out_dir, "masktifs")

    if not os.path.isdir(mask_dir):
        print("SKIP no masktifs:", folder_name)
        continue

    mask_files = sorted(glob.glob(os.path.join(mask_dir, "*_labels.tif")))
    if len(mask_files) == 0:
        print("SKIP no label masks:", folder_name)
        continue

    tile_dir = find_tile_dir(folder)

    png_dir = os.path.join(out_dir, "pebbleplots_pretty")
    os.makedirs(png_dir, exist_ok=True)

    n = min(SAMPLE_PER_FOLDER, len(mask_files))
    pick = list(rng.choice(mask_files, size=n, replace=False))

    print("\nFOLDER:", folder_name, "masks:", len(mask_files), "sample:", n)

    made = 0
    for mf in pick:
        tile_id = os.path.basename(mf).replace("_labels.tif", "")
        src_tile = os.path.join(tile_dir, f"{tile_id}.tif")
        if not os.path.exists(src_tile):
            continue

        out_png = os.path.join(png_dir, f"{folder_name}_{tile_id}_pretty.png")
        if os.path.exists(out_png):
            continue

        with rasterio.open(src_tile) as src:
            img = src.read([1, 2, 3]).transpose(1, 2, 0)

        with rasterio.open(mf) as ms:
            labels = ms.read(1)

        nlab = int(labels.max())
        if nlab == 0:
            continue

        cmap = make_palette_cmap(nlab, rng, alpha=1.0)
        boundaries = find_boundaries(labels, mode="outer")

        boundary_rgba = np.zeros((labels.shape[0], labels.shape[1], 4), dtype=float)
        boundary_rgba[..., 3] = 0.0
        boundary_rgba[boundaries] = (0.0, 0.0, 0.0, OUTLINE_ALPHA)

        plt.figure(figsize=(7, 7))
        plt.imshow(img)
        plt.imshow(labels, cmap=cmap, alpha=OVERLAY_ALPHA, interpolation="nearest")
        plt.imshow(boundary_rgba, interpolation="nearest")
        plt.axis("off")
        plt.title(f"{folder_name} | {tile_id}")
        plt.savefig(out_png, dpi=DPI, bbox_inches="tight")
        plt.close()

        made += 1
        done_pngs_total += 1

    print("  made overlays:", made, "->", png_dir)
    done_folders += 1

print("\nDONE. folders:", done_folders, "total overlays:", done_pngs_total)